# Please type your name and ID here: Adith Srinivasan (adiths)

FINM 37000 - Autumn 2025 - Final Exam

Here are some possibly useful imports and setup. Feel free to add or remove as needed.

In [1]:
import os, sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_dir = os.path.join(repo_root, "src")
if src_dir not in sys.path:
    sys.path.append(src_dir)

In [2]:
import sys
print(sys.executable)

/Users/adithsrinivasan/Documents/GitHub/finm37000-2025/.venv/bin/python


In [3]:
import datetime

In [5]:
import databento as db
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from finm37000 import (
    OptionType,
    add_vol_plot,
    calc_black,
    calc_black_numerical_theta,
    calc_black_one_day_theta,
    calculate_option_vols,
    filter_otm,
    get_databento_api_key,
    get_top_of_book,
    get_all_legs_on,
    temp_env,
    tz_chicago,
)

with temp_env(DATABENTO_API_KEY=get_databento_api_key(f"{repo_root}/.databento_api_key")):
    client = db.Historical()

## 1. (6 points) Futures Trading and Settlement

Suppose you sold 10 [March Corn futures](https://www.cmegroup.com/markets/agriculture/grains/corn.contractSpecs.html) `ZCH6` on December 4th, 2025 for `445'6`.

Assume your margin account is funded appropriately for this trade.

1a. How much does it cost you to put on the trade? Do not include fees or funding your margin account in this cost. Please give you answer in dollars.

**Answer:** $0. Entering a futures contract requires no upfront payment for the contraact itself, outside of posting the margin (which the question asks to ignore), so the cost to put on the trade is $0 (zero dollars).

1b. The `ZCH6` futures settled at `447'2`. How much is your margin account credited or debited? Please indicate a dollar amount and whether it is a debit or a credit.

**Answer:** Since corn futures are quoted in U.S. cents per bushel, a movement from 445'6 (445.75) to 447'2 (447.25) at settlement is a 1.50cent ($0.015) per bushel upswing.

Each corn contract unit is 5,000 bushels. So closing the short position of 10 contracts after this swing results in a -$0.015 x 5,000 = $75 loss per contract.

If I shorted 10 contracts, this is a a **$750 debit to the margin account**.

## 2. (3 points) Cost-of-carry

Calculate the convenience yield for `CLF6` using settlement prices
on December 8th, 2025. You may source spot and settlement prices from any
source you like (e.g., databento), but keep in mind that the CME has recent
settlement prices on the [Crude oil futures product page](https://www.cmegroup.com/markets/energy/crude-oil/light-sweet-crude.settlements.html) and
spot prices are available from the [U.S. Energy Information Administration](https://www.eia.gov/dnav/pet/pet_pri_spt_s1_d.htm).

You may assume the continuously compounded annual interest rate is $4\%$ and the continuously compounded annual storage cost is $2.5\%$.

In [9]:
# Trade date for the exam question
trade_date = datetime.date(2025, 12, 8)

# Pull daily stats for all CL futures on that date
crude_stats, crude_legs = get_all_legs_on(client, trade_date, "CL.FUT")

# Row for CLF6 on 2025-12-08
clf6_row = crude_stats.xs(trade_date).loc["CLF6"]

# Pull raw statistics just for CLF6 on the trade date
raw_stats = client.timeseries.get_range(
    dataset="GLBX.MDP3",
    schema="statistics",
    symbols="CLF6",
    stype_in="raw_symbol",
    start=trade_date,
    end=trade_date + datetime.timedelta(days=1),
).to_df()

# Use the last SETTLEMENT_PRICE record as F
settle_mask = raw_stats["stat_type"] == db.StatType.SETTLEMENT_PRICE
F = float(raw_stats.loc[settle_mask, "price"].iloc[-1])

# Time to expiration T in years, using expiration from crude_stats row
seconds_per_year = 365.25 * 24 * 60 * 60
settlement_time = datetime.time(14, 30)
date_tz = datetime.datetime.combine(trade_date, settlement_time, tzinfo=tz_chicago)
expiration = clf6_row["expiration"]
T = (expiration - date_tz).total_seconds() / seconds_per_year

F, T

(58.88, 0.03000228154232261)

In [12]:
r = 0.04
s = 0.025

S = 59.04 # EIA

# Convenience yield y = r + s - (1/T) * ln(F / S)
y = r + s - (np.log(F / S) / T)
print(f"Convenience yield: {y:.4f} ({y:.2%} per year)")

Convenience yield: 0.1554 (15.54% per year)


**Answer:** Using WTI spot and the CLF6 settlement on 2025‑12‑08 with a 4% continuously compounded interest rate and 2.5% storage cost, the implied convenience yield is **0.1554 (15.54% per year)**.

## 3. (3 points) Theta

Suppose you are long 100 November
[Natural Gas call options](https://www.cmegroup.com/markets/energy/natural-gas/natural-gas.contractSpecs.options.html#optionProductId=1352)
`LNEX5` at strike `3.600` with one day until expiration,
i.e., on Thursday, October 27, 2025. Suppose the underlying price is currently `3.442`, and the continuously
compounded annual interest rate is 4%.

Here is a comparison of the model theoretical values and model-based theta values.
What should risk software report as the one-day theta risk for the position?

In [31]:
atm = 3.442
strike = 3.6
vol = 0.6
interest_rate = 0.04
days_per_year = 365
one_strike = pd.DataFrame(
    {
        "strike_price": strike,
        "days_to_expiration": np.arange(3, 0, -0.25),
        "vol": vol,
    }
)
one_strike["years_to_expiration"] = one_strike["days_to_expiration"] / days_per_year
one_strike["call_theo"] = calc_black(
    F=atm,
    K=one_strike["strike_price"],
    r=interest_rate,
    T=one_strike["years_to_expiration"],
    vol=one_strike["vol"],
    option_type=OptionType.CALL,
)
one_strike["theta"] = (
    calc_black_numerical_theta(
        F=atm,
        K=one_strike["strike_price"],
        r=interest_rate,
        T=one_strike["years_to_expiration"],
        vol=one_strike["vol"],
        option_type=OptionType.CALL,
        dt=0.0001,
    )
    * 1
    / days_per_year
)
one_strike["one_day_theta"] = calc_black_one_day_theta(
    F=atm,
    K=one_strike["strike_price"],
    r=interest_rate,
    T=one_strike["years_to_expiration"],
    vol=one_strike["vol"],
    option_type=OptionType.CALL,
    dt=1 / days_per_year,
)
one_day_tangent = pd.DataFrame(
    {
        "days_to_expiration": one_strike["days_to_expiration"],
        "years_to_expiration": one_strike["years_to_expiration"],
    }
)
tangent_day = 1
tangent_day_mask = np.isclose(one_strike["days_to_expiration"], tangent_day, 0.0001)
tangent_slope = one_strike[tangent_day_mask]["theta"].iloc[0]
tangent_y = one_strike[tangent_day_mask]["call_theo"].iloc[0]
one_day_tangent["tangent"] = (
    tangent_slope * (one_strike["days_to_expiration"] - tangent_day) + tangent_y
)
days_near_exp = 3
near_exp = one_strike[one_strike["days_to_expiration"] < days_near_exp]
near_exp_tangent = one_day_tangent[
    one_day_tangent["days_to_expiration"] < days_near_exp
]
fig = make_subplots(2, 1)

fig.add_trace(
    go.Scatter(
        x=near_exp["days_to_expiration"],
        y=near_exp["theta"],
        mode="lines",
        name="Theta",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=near_exp["days_to_expiration"],
        y=near_exp["call_theo"],
        mode="lines",
        name="Call price",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=near_exp_tangent["days_to_expiration"],
        y=near_exp_tangent["tangent"],
        mode="lines",
        name="T=1 day tangent",
        line=dict(dash="dash"),
    ),
    row=2,
    col=1,
)
fig.update_layout(
    title=f"Theta over time: Strike {strike}, Vol {vol:.1%}",
    xaxis_title="Days to expiration",
    template="plotly_white",
)
fig.update_yaxes(title_text="Theta", row=1, col=1)
fig.update_yaxes(title_text="Price", row=2, col=1)
fig.show()

/Users/adithsrinivasan/Documents/GitHub/finm37000-2025/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

invalid value encountered in sqrt



**Answer:** 

- One-day model theta risk for the position = 0.0078 x 100 = **0.78** (i.e., option price units before the contract multiplier).

- This corresponds to **$7,800** when multiplied by the CME Natural Gas option multiplier of 10,000.

## 4. (6 points) Skew

Here are natural gas option skews for two expirations (market-implied, not fit, using OTM options).
I have used the American ON options to imply the volatility.

Answers below...

In [25]:
interest_rate = 0.04
vol_target = "iv_midprice"
days_per_year = 365
start = pd.Timestamp("2025-10-24T09:59:00", tz=tz_chicago)
end = pd.Timestamp("2025-10-24T10:00:00", tz=tz_chicago)
parent_option = "ON"
option_expirations = {
    "NGF26": f"{parent_option}F6",
    "NGG26": f"{parent_option}G6",
}
underlying_symbols = list(option_expirations.keys())

In [26]:
all_options = client.timeseries.get_range(
    dataset=db.Dataset.GLBX_MDP3,
    schema="definition",
    symbols=f"{parent_option}.OPT",
    stype_in="parent",
    start=start.date(),
).to_df()

In [27]:
options_chain = all_options[all_options["underlying"].isin(underlying_symbols)].copy()
options_chain = options_chain.sort_values("strike_price")

In [28]:
top_prices = {}
for underlying_symbol in underlying_symbols:
    chain_slice = options_chain[options_chain["underlying"] == underlying_symbol]
    df = get_top_of_book(
        symbols=[underlying_symbol, *chain_slice["raw_symbol"]],
        start=start,
        end=end,
        client=client,
    )
    df["underlying"] = underlying_symbol
    top_prices[underlying_symbol] = df

In [29]:
underlying_price = {}
otm_options = {}
all_options = {}
options_chain["years_to_expiration"] = (
    (options_chain["expiration"] - end).dt.total_seconds()
    / days_per_year
    / 24
    / 60
    / 60
)
for underlying_symbol in underlying_symbols:
    top_price = top_prices[underlying_symbol].drop("underlying", axis=1)
    partial_chain = options_chain[
        options_chain["underlying"] == underlying_symbol
    ].copy()
    with_vol, underlying_price[underlying_symbol] = calculate_option_vols(
        top_price, underlying_symbol, partial_chain, interest_rate
    )
    otm_options[underlying_symbol] = filter_otm(
        with_vol, underlying_price[underlying_symbol]
    )
    all_options[underlying_symbol] = with_vol
    df = otm_options[underlying_symbol]

Cannot find OptionType.CALL vol between lb=1e-05 and ub=4 at strike 0.25: lower_vol=3.971403951384679 target=3.9989999999999997 upper_vol=3.995337336411769
  F=4.2490000000000006 T=0.1731164383561644 r=0.04 mid=3.9989999999999997


In [30]:
fig = go.Figure()
for underlying_symbol in underlying_symbols:
    add_vol_plot(
        fig=fig,
        vol_df=otm_options[underlying_symbol],
        name=f"OTM {option_expirations[underlying_symbol]} bid vol",
        y_col="iv_bid",
    )
    add_vol_plot(
        fig=fig,
        vol_df=otm_options[underlying_symbol],
        name=f"OTM {option_expirations[underlying_symbol]} ask vol",
        y_col="iv_ask",
    )
fig.update_layout(
    title=f"Implied Volatility by Strike - {end}",
    xaxis_title="Strike Price",
    yaxis_title="Implied Volatility (σ)",
    template="plotly_white",
)
fig.update_yaxes(tickformat=".0%")
fig.show()

a. Is there a calendar spread arbitrage? Justify your answer.


**Answer:** In Black76, call price is strictly increasing w.r.t. total variance. We can use this to restate the calendar spread arbitrage rule using implied vol or total variance...

Starting from the classic calendar-arbitrage rule:

$$
C(K, T_2) \ge C(K, T_1),
$$

we obtain the **no-calendar-arbitrage condition on implied vols**:

$$
\sigma(T_2, K) \ge \sigma(T_1, K)\sqrt{\frac{T_1}{T_2}},
$$

which is exactly the requirement that **total variance must be non-decreasing in maturity**:

$$
w(T_2, K) \ge w(T_1, K).
$$

EVIDENCE FROM GRAPH: This is evident from the graph, where ONG6 bid and ask vols are higher than ONF6 bid and ask vols.

EVIDENCE FROM DATA: The code below also directly tests this by comparing the front (NGF26) and back (NGG26) natural‑gas option expiries at the same strikes.

It builds “mid” implied vols from bid/ask for each strike to smooth bid/ask noise, then converts them to total variance using each expiry’s time to expiration: $$w(T,K) = \sigma(T,K)^2 T$$

For every common strike it computes the total variance for the back (Feb) and front (Jan) options and calculates the difference; if this is never negative, total variance is non‑decreasing in maturity, which is exactly the no–calendar‑arbitrage condition in terms of implied vols.

In [ ]:
# Front (Jan) vs back (Feb) NG options
front = otm_options["NGF26"].set_index("strike_price")
back  = otm_options["NGG26"].set_index("strike_price")

# Use mid vols to smooth bid/ask noise
front_mid = 0.5 * (front["iv_bid"] + front["iv_ask"])
back_mid  = 0.5 * (back["iv_bid"] + back["iv_ask"])

# Common strikes
common_strikes = front.index.intersection(back.index)

# Time to expiration (constant within each expiry)
T_front = float(front.loc[common_strikes, "years_to_expiration"].iloc[0])
T_back  = float(back.loc[common_strikes, "years_to_expiration"].iloc[0])

# Total variance w = vol^2 * T for each expiry
var_front = (front_mid.loc[common_strikes] ** 2) * T_front
var_back  = (back_mid.loc[common_strikes]  ** 2) * T_back

diff = var_back - var_front

print("Min(var_back - var_front) =", diff.min())
print("Any strikes with calendar arb (var_back < var_front)?", (diff < 0).any())


Min(var_back - var_front) = 0.020905852715176393
Any strikes with calendar arb (var_back < var_front)? False


...`Any strikes with calendar arb (var_back < var_front)? False` confirms there are no strikes where total variance decreases with maturity, so the term structure is calendar‑arbitrage free.

b. Would you rather capture a calendar spread arbitrage with European `LNE` options or American `ON` options? Why?


**Answer:** American ON options contain early-exercise value, so differences in their implied vols across expiries may reflect exercise optionality rather than true mispricing. European options remove this complication, so any calendar-spread violation in LNE options represents a clean, tradable arbitrage.